# LightRAG Evaluation Pipeline — Google Colab

Runs the full eval pipeline (ingest → query all modes → judge → report).
Uses **OpenAI** for LLM, embeddings, and judging. CPU runtime is fine — all inference
happens in OpenAI's cloud.

### What you need
1. Your **OpenAI API key**
2. Your **document corpus** (`.txt` / `.md` files)
3. Your **multihop questions** in the JSON format shown in Cell 4

### Runtime
Set `Runtime → Change runtime type → CPU` — no GPU required.

---
## Step 1 — Clone repo and install dependencies

In [ ]:
import os

if not os.path.exists('LightRAG'):
    !git clone https://github.com/anasmostafa23/LightRAG.git

os.chdir('LightRAG')
print('Working directory:', os.getcwd())

# offline-llm includes the openai package; nest_asyncio lets asyncio.run() work in Colab
!pip install -q -e ".[offline-llm]" nest_asyncio

---
## Step 2 — Configuration

**Edit this cell** before running anything else.

In [ ]:
import os

# ── API Key ────────────────────────────────────────────────────────────────
# Option A: paste directly (fine for personal notebooks)
OPENAI_API_KEY = "sk-..."   # ← your key here

# Option B: Colab Secrets (recommended for shared notebooks)
# 1. Click the 🔑 icon in the left sidebar → add secret OPENAI_API_KEY
# 2. Uncomment the two lines below and delete Option A
# from google.colab import userdata
# OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# ── Models ─────────────────────────────────────────────────────────────────
# LLM for indexing + querying
#   gpt-4o-mini  → cheapest, good quality  (~$0.15/1M input tokens)
#   gpt-4o       → best quality            (~$2.50/1M input tokens)
LLM_MODEL = "gpt-4o-mini"

# Embedding model — MUST stay the same for all runs on the same index
#   text-embedding-3-small → 1536-dim, cheapest  (~$0.02/1M tokens)
#   text-embedding-3-large → 3072-dim, best      (~$0.13/1M tokens)
EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM   = 1536   # must match EMBED_MODEL (3072 for large)

# Judge model — scores each answer (can be different from LLM_MODEL)
JUDGE_MODEL = "gpt-4o-mini"

# ── Query modes to evaluate ────────────────────────────────────────────────
MODES = ["naive", "local", "global", "hybrid", "mix"]

# ── Paths ──────────────────────────────────────────────────────────────────
WORKING_DIR    = "/content/LightRAG/eval/rag_storage"
CORPUS_DIR     = "/content/LightRAG/eval/corpus"
QUESTIONS_FILE = "/content/LightRAG/eval/questions.json"
RESULTS_DIR    = "/content/LightRAG/eval/results"

# ── Pipeline flags ─────────────────────────────────────────────────────────
SKIP_INGESTION = False   # Set True on re-runs to reuse the existing index
FORCE_REINDEX  = False   # Set True to wipe + rebuild the index from scratch

assert OPENAI_API_KEY.startswith('sk-'), 'Paste your OpenAI API key above!'
print('Config OK')

---
## Step 3 — Upload your documents

Click the file picker, select your `.txt` or `.md` files.

In [ ]:
import os
from google.colab import files

os.makedirs(CORPUS_DIR, exist_ok=True)

print(f'Select your .txt or .md files → they will be saved to {CORPUS_DIR}')
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = os.path.join(CORPUS_DIR, fname)
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  Saved {dest} ({len(content):,} bytes)')

all_docs = [f for f in os.listdir(CORPUS_DIR) if f.endswith(('.txt', '.md'))]
print(f'\nCorpus: {len(all_docs)} file(s) → {all_docs}')

---
## Step 4 — Define your multihop questions

Edit the list below. Required fields: `id`, `question`, `ground_truth`.

In [ ]:
import json, os

questions = [
    {
        "id": "q1",
        "hop_count": 2,
        "question": "REPLACE: 2-hop question requiring two documents to answer.",
        "ground_truth": "REPLACE: Complete answer covering both reasoning hops.",
    },
    {
        "id": "q2",
        "hop_count": 3,
        "question": "REPLACE: 3-hop question chaining facts from three documents.",
        "ground_truth": "REPLACE: Complete answer covering all three hops.",
    },
]

os.makedirs(os.path.dirname(QUESTIONS_FILE), exist_ok=True)
with open(QUESTIONS_FILE, 'w') as f:
    json.dump({'questions': questions}, f, indent=2)

print(f'Saved {len(questions)} question(s) to {QUESTIONS_FILE}')

---
## Step 5 — Run the pipeline

This cell does everything:
1. Ingests documents → builds the knowledge graph
2. Queries each question across all selected modes
3. Scores each answer with the judge model (4 metrics, 1–5 scale)
4. Prints a summary table and saves CSV + JSON to `eval/results/`

In [ ]:
import sys, asyncio, nest_asyncio
nest_asyncio.apply()

REPO_ROOT = '/content/LightRAG'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from eval.config import EvalConfig
from eval.pipeline import run

cfg = EvalConfig(
    backend           = 'openai',
    openai_api_key    = OPENAI_API_KEY,
    openai_llm_model  = LLM_MODEL,
    openai_embed_model= EMBED_MODEL,
    openai_embed_dim  = EMBED_DIM,
    judge_model       = JUDGE_MODEL,
    modes             = MODES,
    working_dir       = WORKING_DIR,
    corpus_dir        = CORPUS_DIR,
    questions_file    = QUESTIONS_FILE,
    results_dir       = RESULTS_DIR,
    skip_ingestion    = SKIP_INGESTION,
    force_reindex     = FORCE_REINDEX,
)

asyncio.run(run(cfg))

---
## Step 6 — Explore and download results

In [ ]:
import pandas as pd, glob, os

csv_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.csv')))
if not csv_files:
    print('No results yet — run Step 5 first.')
else:
    df = pd.read_csv(csv_files[-1])
    score_cols = ['faithfulness', 'answer_relevance', 'completeness', 'coherence', 'overall']

    print('=== Mean scores per mode (1–5) ===')
    display(df.groupby('mode')[score_cols].mean().round(2))

    print('\n=== Full results ===')
    display(df[['question_id','hop_count','mode','overall','latency_s','answer']].head(40))

In [ ]:
from google.colab import files as colab_files
import glob, os

for path in sorted(glob.glob(os.path.join(RESULTS_DIR, '*')))[-2:]:
    print(f'Downloading {path}')
    colab_files.download(path)

---
## Tips

**Re-run queries without re-indexing** (saves time + cost after first run):
```python
SKIP_INGESTION = True
```

**Only evaluate specific modes** (e.g. skip naive/global):
```python
MODES = ['hybrid', 'mix']
```

**Use a smarter judge** without changing the indexing model:
```python
JUDGE_MODEL = 'gpt-4o'   # better reasoning, slightly more expensive
```

**Cost estimate** (rough, depends on corpus/question size):
| Phase | Model | Approx cost |
|---|---|---|
| Indexing (entity extraction) | gpt-4o-mini | $0.01–$0.10 per doc |
| Embedding | text-embedding-3-small | < $0.01 total |
| Querying (5 modes × N questions) | gpt-4o-mini | $0.01–$0.05 per question |
| Judging (5 modes × N questions) | gpt-4o-mini | $0.01–$0.03 per answer |

**Scoring dimensions** (1–5):
| Metric | Measures |
|---|---|
| faithfulness | No hallucinations |
| answer_relevance | Directly answers the question |
| completeness | All multihop reasoning steps covered |
| coherence | Logically structured answer |